In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")

In [3]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery",
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [6]:
DOMAIN_PHRASES = {
    "mystery_1": {
        "actions": {
            "attack": "attack",
            "succumb": "succumb",
            "overcome": "overcome",
            "feast": "feast"
        },
        "predicates": {
            "planet": "planet",
            "province": "province",
            "harmony": "harmony",
            "craves": "craves",
            "pain": "pain"
        }
    },
    "mystery_2": {
        "actions": {
            "attack": "illuminate",
            "succumb": "silence",
            "overcome": "distill",
            "feast": "divest"
        },
        "predicates": {
            "planet": "aura",
            "province": "essence",
            "harmony": "nexus",
            "craves": "harmonizes",
            "pain": "pulse"
        }
    },
}

In [7]:
def extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=True, min_pos=None):
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            if min_pos is not None and p < min_pos:
                continue
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

In [8]:
from vllm import LLM

llm = LLM(model=model_id, tensor_parallel_size=4, enforce_eager=True, max_seq_len_to_capture=20000, max_num_batched_tokens=3000)

INFO 05-05 00:16:47 __init__.py:190] Automatically detected platform cuda.
INFO 05-05 00:17:00 config.py:542] This model supports multiple tasks: {'embed', 'reward', 'score', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 05-05 00:17:01 config.py:1401] Defaulting to use mp for distributed inference
WARNING 05-05 00:17:01 arg_utils.py:1135] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 05-05 00:17:01 config.py:1556] Chunked prefill is enabled with max_num_batched_tokens=3000.
WARNING 05-05 00:17:01 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 05-05 00:17:01 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 05-05 00:17:01

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


INFO 05-05 00:17:07 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1325626) INFO 05-05 00:17:07 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1325618) INFO 05-05 00:17:07 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1325621) INFO 05-05 00:17:07 cuda.py:230] Using Flash Attention backend.
INFO 05-05 00:17:08 utils.py:950] Found nccl from library libnccl.so.2
INFO 05-05 00:17:08 pynccl.py:69] vLLM is using nccl==2.21.5
(VllmWorkerProcess pid=1325626) (VllmWorkerProcess pid=1325621) (VllmWorkerProcess pid=1325618) INFO 05-05 00:17:08 utils.py:950] Found nccl from library libnccl.so.2
INFO 05-05 00:17:08 utils.py:950] Found nccl from library libnccl.so.2
INFO 05-05 00:17:08 utils.py:950] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=1325621) (VllmWorkerProcess pid=1325626) (VllmWorkerProcess pid=1325618) INFO 05-05 00:17:08 pynccl.py:69] vLLM is using nccl==2.21.5
INFO 05-05 00:17:08 pynccl.py:69] vLLM is using nc

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


INFO 05-05 00:17:19 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=1325621) INFO 05-05 00:17:19 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=1325618) INFO 05-05 00:17:19 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=1325626) INFO 05-05 00:17:20 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=1325621) INFO 05-05 00:17:23 worker.py:267] Memory profiling takes 3.26 seconds
(VllmWorkerProcess pid=1325621) INFO 05-05 00:17:23 worker.py:267] the current vLLM instance can use total_gpu_memory (79.10GiB) x gpu_memory_utilization (0.90) = 71.19GiB
(VllmWorkerProcess pid=1325621) INFO 05-05 00:17:23 worker.py:267] model weights take 15.39GiB; non_torch_memory takes 4.13GiB; PyTorch activation peak memory takes 0.29GiB; the rest of the memory reserved for KV Cache is 51.37GiB.
(VllmWorkerProcess pid=1325618) INFO 05-05 00:17:23 worker.py:267] Memory pro

In [9]:
repr_file = f"mystery_representations_greedy/mystery_{1}/mean_reprs_mystery_{1}.json"

In [10]:
with open(repr_file, 'r') as f:
        reprs = json.load(f)
    
mean_reprs = {k: np.array(v) for k, v in reprs["mean_reprs"].items()}
mean_actions = np.array(reprs["mean_actions"])
mean_predicates = np.array(reprs["mean_predicates"])

# Get domain phrases
domain_key = f"mystery_{1}"
phrases = DOMAIN_PHRASES[domain_key]
action_phrases = list(phrases["actions"].values())


phrases = list(phrases["actions"].values()) + list(phrases["predicates"].values())


In [11]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    max_tokens=1,
    temperature=0,
    top_k=1,
    seed=0,
)

In [12]:
# Load tokenizer
tokenizer = initialize_tokenizer(model_id)
    
# Load dataset
dataset_name = f"dmitriihook/qwq-32b-planning-mystery-{1}-24k-greedy"

dataset = load_dataset(dataset_name)["train"]

In [13]:
all_tokens = []
phrase_masks = {phrase: [] for phrase in phrases}

In [14]:
row_ids = list(range(25))

row_ids = [
    x for x in row_ids for _ in range(3)
]

# row_ids = [

row_ids = list(range(15))

initial_lines = 60

for i in row_ids:
    row = dataset[i]
    
    # Process text
    # text = "\n\n".join(row["generation"].split("\n\n")[:initial_lines])
    tokens = tokenize_blocksworld_generation(tokenizer, row)[:, :-2][0]
    tokens = tokens[:3000]
    all_tokens.append(tokens)
    
    # Get phrase positions for this row
    phrase_positions = {
        phrase: extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=False)
        for phrase in phrases
    }
    
    # Create phrase masks for this row
    row_phrase_masks = {
        phrase: np.zeros(tokens.shape[0])
        for phrase in phrases
    }
    
    for phrase in phrases:
        positions = phrase_positions[phrase]
        for start, end in positions:
            row_phrase_masks[phrase][start:end] = 1
    
    # Add masks to batch
    for phrase in phrases:
        phrase_masks[phrase].append(row_phrase_masks[phrase])

masks_combined = {
    k: np.concatenate(v, axis=0) for k, v in phrase_masks.items()
}

combined_len = masks_combined[phrases[0]].shape[0] if phrases else 0

In [33]:
def create_hook(phrases, action_phrases, masks_batch_combined, mean_reprs, mean_actions, mean_predicates, combined_len, block_size=4096, scale=1):
    def hook(module, input, output):
        meta = getattr(module, "_meta", {})
        meta["mask_offset"] = meta.get("mask_offset", 0)
        
        # if input[0].shape[0] > 200:
        #     print(input[0].shape[0], meta["mask_offset"])
        
        if meta["mask_offset"] >= combined_len:
            return output
        
        mask_start = meta["mask_offset"]
        mask_end = mask_start + block_size
        
        meta["mask_offset"] = mask_end
        module._meta = meta
        
        hs, res = output

        if hs.device == torch.device("cuda:0"):
            steering_mask = masks_batch_combined[phrases[0]]
            with open("logging_file_single.txt", "a") as f:
                f.write(f"{hs.cpu().float().numpy().tolist()[2000][:2000]}\n")
                # f.write(f"{input[0].cpu().numpy().tolist()}\n")
                # f.write(
                #     f"{hs.shape}, {steering_mask[mask_start:mask_end].shape} {len(input[0].cpu().numpy().tolist())}\n"
                # )
                # f.write(f"{input[3]}\n")
                # f.write(f"{input[3].slot_mapping.cpu().numpy().tolist()}\n")
      
        for ip, phrase in enumerate(phrases):
            if phrase in action_phrases:
                adjustment = mean_actions
            else:
                adjustment = mean_predicates
            
            steering_mask = masks_batch_combined[phrase]
            
            steering_mask = steering_mask[mask_start:mask_end]
            steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
            
            steering_vector = mean_reprs[phrase] - adjustment
            steering_vector = steering_mask[:, None] * steering_vector
        
            steering_mask = torch.tensor(steering_mask[:, None], dtype=torch.int32, device=hs.device)
            steering_vector = torch.tensor(steering_vector, dtype=hs.dtype, device=hs.device)
            
            a = 1 / (1 + scale)
            b = 1 - a
            
            hs = torch.where(steering_mask == 0, hs, steering_vector)
        return hs, res
    
    return hook

In [34]:
# Create a hook function using the factory
current_hook = create_hook(
    phrases=phrases,
    action_phrases=action_phrases,
    masks_batch_combined=masks_combined,
    mean_reprs=mean_reprs,
    mean_actions=mean_actions,
    mean_predicates=mean_predicates,
    combined_len=combined_len,
    block_size=3000,
    scale=1,
)

In [35]:
from collections import OrderedDict

def add_hook(module, hook_fn):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    module.register_forward_hook(hook_fn)
    

In [36]:
# Apply hook to model
llm.apply_model(
    lambda x: add_hook(x.model.layers[47], current_hook),
)

[None, None, None, None]

In [37]:
from vllm import TokensPrompt
prompts = [TokensPrompt(prompt_token_ids=tokens.tolist()) for tokens in all_tokens]

In [38]:
# results = llm.generate(prompts, sampling_params=sampling_params)
for p in prompts:
    llm.generate([p], sampling_params=sampling_params)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s, est. speed input: 3264.80 toks/s, output: 1.09 toks/s]
